In [ ]:
from langchain_experimental.tools import PythonREPLTool

python_tool = PythonREPLTool()

result = python_tool.invoke("print(2 + 2); import math; print(math.sqrt(16))")
print(result)

print(python_tool.name)      
print(python_tool.description) # 'A Python shell...'

4
4.0

Python_REPL
A Python shell. Use this to execute python commands. Input should be a valid python command. If you want to see the output of a value, you should print it out with `print(...)`.
{'query': {'title': 'Query', 'type': 'string'}}


In [5]:
!pip install -q youtube-search

from langchain_community.tools import YouTubeSearchTool

youtube_tool = YouTubeSearchTool()

result = youtube_tool.invoke("Python programming tutorial")
print(result)

print(youtube_tool.name)        # 'youtube_search'
print(youtube_tool.description)

['https://www.youtube.com/watch?v=_uQrJ0TkZlc&pp=ygUbUHl0aG9uIHByb2dyYW1taW5nIHR1dG9yaWFs', 'https://www.youtube.com/watch?v=K5KVEU3aaeQ&pp=ygUbUHl0aG9uIHByb2dyYW1taW5nIHR1dG9yaWFs']
youtube_search
search for youtube videos associated with a person. the input to this tool should be a comma separated list, the first part contains a person name and the second a number that is the maximum number of video results to return aka num_results. the second part is optional


Custom Tools

In [6]:
from langchain_core.tools import tool

In [7]:
def multiply(a, b):
    """Multiply two numbers"""
    return a*b

In [ ]:
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [9]:
@tool
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [10]:
result = multiply.invoke({"a":3, "b":5})
print(result)

15


In [11]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


Method 2 - Using StructuredTool

In [12]:
from langchain.tools import StructuredTool
from pydantic import BaseModel, Field

In [13]:
class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

/tmp/ipykernel_21680/3279971488.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a: int = Field(required=True, description="The first number to add")
/tmp/ipykernel_21680/3279971488.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b: int = Field(required=True, description="The second number to add")


In [14]:
def multiply_func(a: int, b: int) -> int:
    return a * b

In [15]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [16]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'required': True, 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'required': True, 'title': 'B', 'type': 'integer'}}


Method 3 - Using BaseTool Class

In [17]:
from langchain.tools import BaseTool
from typing import Type

In [18]:
class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

/tmp/ipykernel_21680/3279971488.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a: int = Field(required=True, description="The first number to add")
/tmp/ipykernel_21680/3279971488.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b: int = Field(required=True, description="The second number to add")


In [19]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [20]:
multiply_tool = MultiplyTool()

In [21]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'required': True, 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'required': True, 'title': 'B', 'type': 'integer'}}


ToolKits

In [22]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

In [23]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]

In [24]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)

add => Add two numbers
multiply => Multiply two numbers
